In [ ]:
import kagglehub
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
import os
# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
csv_file_path = os.path.join(path, "Q1_data.csv")
df = pd.read_csv(csv_file_path)

In [ ]:
# Task 2: Write your code here:
print("\n--- First 5 rows of the data ---")
print(df.head())

In [ ]:
# Task 3: Write your code here:
print("\n--- Dataset Information ---")
df.info()

In [ ]:
# Task 4: Write your code here:
print("\n--- Statistical Description ---")
print(df.describe())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Task 5: Write your code here:
plt.figure(figsize=(8, 5))
sns.histplot(df['Delivery_Time'], kde=True, color='green')
plt.title('Distribution of Delivery Time')
plt.xlabel('Delivery Time (minutes)')
plt.ylabel('Count')
plt.show()

In [ ]:
# Task 1: Write your code here:
from sklearn.preprocessing import StandardScaler
import pandas as pd

if 'Order_ID' in df.columns:
    df = df.drop(columns=['Order_ID'])
    print("Column 'Order_ID' dropped.")

In [ ]:
# Task 2: Write your code here:
print("Missing values before cleaning:")
print(df.isnull().sum())

initial_rows = df.shape[0]
df.dropna(subset=['Delivery_Time'], inplace=True)
print(f"Dropped {initial_rows - df.shape[0]} rows with missing 'Delivery_Time'.")
for column in ['Weather', 'Traffic_Level', 'Time_of_Day']:
    if column in df.columns and df[column].isnull().any():
        mode_val = df[column].mode()[0]
        df[column].fillna(mode_val, inplace=True)
        print(f"Imputed missing values in '{column}' with mode: {mode_val}")

if df['Courier_Experience_yrs'].isnull().any():
    median_val = df['Courier_Experience_yrs'].median()
    df['Courier_Experience_yrs'].fillna(median_val, inplace=True)
    print(f"Imputed missing values with median:: {median_val}")
else:
    print("has no missing values or already handled")



print("\nMissing values after handling::")
print(df.isnull().sum())

duplicate_count = df.duplicated().sum()
print(f"\nFound {duplicate_count} duplicate rows.")
if duplicate_count > 0:
    df = df.drop_duplicates()
    print("Duplicates removed.")

In [ ]:
# Task 3: Write your code here:
# Encode categorical variables using One-Hot Encoding
df = pd.get_dummies(df, drop_first=True)
print("\nCategorical variables encoded using One-Hot Encoding.")

In [ ]:
# Task 4: Write your code here:
scaler = StandardScaler()

features = df.drop(columns=['Delivery_Time'])
target = df['Delivery_Time']

scaled_features = scaler.fit_transform(features)
df_cleaned = pd.DataFrame(scaled_features, columns=features.columns)
df_cleaned['Delivery_Time'] = target.values

print("\nFeature scaling finished using StandardScaler.")

In [ ]:
# Task 5: Write your code here:
scaler = StandardScaler()


features = df.drop(columns=['Delivery_Time'])
target = df['Delivery_Time']

scaled_features = scaler.fit_transform(features)

df_cleaned = pd.DataFrame(scaled_features, columns=features.columns)
df_cleaned['Delivery_Time'] = target.values

print("\nFeature scaling finished using StandardScaler.")

In [ ]:
# Task 6: Write your code here:
print("\n--- Target Data Check ---")
print(df_cleaned['Delivery_Time'].describe())
print("\nFinal data shape:", df_cleaned.shape)
print(df_cleaned.head())

In [ ]:
# Task 1: Write your code here:

from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
import numpy as np

X = df_cleaned.drop(columns=['Delivery_Time'])
y = df_cleaned['Delivery_Time']

print("Dataset split into features (X) and target (y).")
print("Shape of X:", X.shape)
print("Shape of y:", y.shape)

In [ ]:
# Task 2,3,4,5: Write your code here:

kf = KFold(n_splits=5, shuffle=True, random_state=42)

mae_scores = []

print("Starting model training with K-Fold...")

# 3-Train a RandomForest:
fold_number = 1
for train_index, test_index in kf.split(X):
    X_train_fold, X_test_fold = X.iloc[train_index], X.iloc[test_index]
    y_train_fold, y_test_fold = y.iloc[train_index], y.iloc[test_index]



    model = RandomForestRegressor(n_estimators=100, random_state=42)

    model.fit(X_train_fold, y_train_fold)

    predictions = model.predict(X_test_fold)

    # task-4:
    mae = mean_absolute_error(y_test_fold, predictions)
    mae_scores.append(mae)

    print(f"Fold {fold_number} MAE: {mae:.4f}")
    fold_number += 1


average_mae = np.mean(mae_scores)
print("\n--- Final Evaluation ---")
print(f"Average MAE across all folds: {average_mae:.4f}")

In [ ]:
# Task 1: Write your code here:
import matplotlib.pyplot as plt
import seaborn as sns

importances = model.feature_importances_
feature_names = features.columns


importance_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances})

importance_df = importance_df.sort_values(by='Importance', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x='Importance', y='Feature', data=importance_df)
plt.title('Feature Importance (What affects delivery time most?)')
plt.xlabel('Importance Score')
plt.ylabel('Features')
plt.show()



In [ ]:
# Task 2: Write your code here:
predicted_times = model.predict(X_test_fold)

plt.figure(figsize=(10, 6))
sns.histplot(predicted_times, kde=True, bins=30)

plt.title('Histogram of Predicted Delivery Times')
plt.xlabel('Predicted Time (minutes)')
plt.ylabel('Frequency')
plt.show()

print("\nSample:")
for i in range(5):
    print(f"Actual: {y_test_fold.iloc[i]:.2f} mins | Predicted: {predicted_times[i]:.2f} mins")

In [ ]:
# Task Bonus: Write your code here:
!pip install catboost
from catboost import CatBoostRegressor

ensemble_mae_scores = []

print("Starting ensemble model training with K-Fold...")

for train_index, test_index in kf.split(X):
    X_train_fold, X_test_fold = X.iloc[train_index], X.iloc[test_index]
    y_train_fold, y_test_fold = y.iloc[train_index], y.iloc[test_index]

    # model 1:
    rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
    rf_model.fit(X_train_fold, y_train_fold)
    rf_predictions = rf_model.predict(X_test_fold)

    # model 2:
    cb_model = CatBoostRegressor(iterations=100, learning_rate=0.1, depth=6, random_seed=42, verbose=False)
    cb_model.fit(X_train_fold, y_train_fold)
    cb_predictions = cb_model.predict(X_test_fold)
    averaged_predictions = (rf_predictions + cb_predictions) / 2

    ensemble_mae = mean_absolute_error(y_test_fold, averaged_predictions)
    ensemble_mae_scores.append(ensemble_mae)



    print(f"Fold {len(ensemble_mae_scores)} Ensemble MAE: {ensemble_mae:.4f}")

average_ensemble_mae = np.mean(ensemble_mae_scores)
print("\n--- Final Ensemble Evaluation ---")
print(f"Average Ensemble MAE across all folds: {average_ensemble_mae:.4f}")